In [1]:
!pip install pandas
!pip install ipython-sql prettytable 

import prettytable

prettytable.DEFAULT = 'DEFAULT'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 160.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 152.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.8/769.8 kB 52.8 MB/s eta 0:00:00
    Found existing installation: wcwidth 0.2.13
    Uninstalling wcwidth-0.2.13:
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ipython-sql]


### Store the datasets in database tables

To analyze the data using SQL, it first needs to be loaded into SQLite DB.
We will create three tables in as under:

1.  **CENSUS_DATA**
2.  **CHICAGO_PUBLIC_SCHOOLS**
3.  **CHICAGO_CRIME_DATA**


Load the `pandas` and `sqlite3` libraries and establish a connection to `FinalDB.db`


In [3]:
import pandas as pd 
import sqlite3 
conn = sqlite3.connect('FinalDB.db')

Load the SQL magic module


In [4]:
%load_ext sql

Use `Pandas` to load the data available in the links above to dataframes. Use these dataframes to load data on to the database `FinalDB.db` as required tables.


In [17]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv"
census_df = pd.read_csv(url)

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv"
School_df = pd.read_csv(url)

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCrimeData.csv"
crime_df = pd.read_csv(url)


census_df.to_sql("CENSUS_DATA", conn, if_exists="replace", index=False)

School_df.to_sql("CHICAGO_PUBLIC_SCHOOLS", conn, if_exists="replace", index=False)

crime_df.to_sql("CHICAGO_CRIME_DATA", conn, if_exists="replace", index=False)


533

Establish a connection between SQL magic module and the database `FinalDB.db`


In [25]:

%sql sqlite:///FinalDB.db


### Problem 1

##### Find the total number of crimes recorded in the CRIME table.


In [61]:
%%sql
SELECT COUNT (*) FROM CHICAGO_CRIME_DATA;


 * sqlite:///FinalDB.db
Done.

COUNT (*)
533


### Problem 2

##### List community area names and numbers with per capita income less than 11000.


In [100]:
%%sql
Select COMMUNITY_AREA_NAME, COMMUNITY_AREA_NUMBER FROM CENSUS_DATA
WHERE PER_CAPITA_INCOME < 11000 ;


 * sqlite:///FinalDB.db
Done.

COMMUNITY_AREA_NAME,COMMUNITY_AREA_NUMBER
West Garfield Park,26.0
South Lawndale,30.0
Fuller Park,37.0
Riverdale,54.0


### Problem 3

##### List all case numbers for crimes involving minors?(children are not considered minors for the purposes of crime analysis) 


In [52]:
%%sql
SELECT CASE_NUMBER FROM CHICAGO_CRIME_DATA
WHERE DESCRIPTION LIKE '%minor%' ;

 * sqlite:///FinalDB.db
Done.

CASE_NUMBER
HL266884
HK238408


### Problem 4

##### List all kidnapping crimes involving a child?


In [65]:
%%sql
SELECT CASE_NUMBER FROM CHICAGO_CRIME_DATA
WHERE PRIMARY_TYPE = "KIDNAPPING" AND DESCRIPTION LIKE '%child%' ;

 * sqlite:///FinalDB.db
Done.

CASE_NUMBER
HN144152


### Problem 5

##### List the kind of crimes that were recorded at schools. (No repetitions)


In [70]:
%%sql
SELECT DISTINCT PRIMARY_TYPE FROM CHICAGO_CRIME_DATA
WHERE LOCATION_DESCRIPTION LIKE "%SCHOOL%"
    ;

 * sqlite:///FinalDB.db
Done.

PRIMARY_TYPE
BATTERY
CRIMINAL DAMAGE
NARCOTICS
ASSAULT
CRIMINAL TRESPASS
PUBLIC PEACE VIOLATION


### Problem 6

##### List the type of schools along with the average safety score for each type.


In [102]:
%%sql 
SELECT "Elementary, Middle, or High School"AS School_Type,
       AVG(SAFETY_SCORE) AS Average_safety_score
FROM CHICAGO_PUBLIC_SCHOOLS
GROUP BY "Elementary, Middle, or High School";

 * sqlite:///FinalDB.db
Done.

School_Type,Average_safety_score
ES,49.52038369304557
HS,49.62352941176471
MS,48.0


### Problem 7

##### List 5 community areas with highest % of households below poverty line


In [87]:
%%sql
SELECT COMMUNITY_AREA_NAME, PERCENT_HOUSEHOLDS_BELOW_POVERTY 
FROM CENSUS_DATA
ORDER BY PERCENT_HOUSEHOLDS_BELOW_POVERTY DESC
LIMIT 5


 * sqlite:///FinalDB.db
Done.

COMMUNITY_AREA_NAME,PERCENT_HOUSEHOLDS_BELOW_POVERTY
Riverdale,56.5
Fuller Park,51.2
Englewood,46.6
North Lawndale,43.1
East Garfield Park,42.4


### Problem 8

##### Which community area is most crime prone? Display the coumminty area number only.


In [97]:
%%sql
SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_CRIME_DATA 
    Group BY COMMUNITY_AREA_NUMBER
    Order BY count(*) desc
    limit 1;

 * sqlite:///FinalDB.db
Done.

COMMUNITY_AREA_NUMBER
25.0



Double-click **here** for a hint

<!--
Query for the 'community area number' that has most number of incidents
-->


### Problem 9

##### Use a sub-query to find the name of the community area with highest hardship index


In [98]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE HARDSHIP_INDEX = (
    SELECT MAX(HARDSHIP_INDEX)
    FROM CENSUS_DATA
);


 * sqlite:///FinalDB.db
Done.

COMMUNITY_AREA_NAME
Riverdale


### Problem 10

##### Use a sub-query to determine the Community Area Name with most number of crimes?


In [103]:
%%sql
SELECT COMMUNITY_AREA_NAME
FROM CENSUS_DATA
WHERE COMMUNITY_AREA_NUMBER = (
    SELECT COMMUNITY_AREA_NUMBER
    FROM CHICAGO_CRIME_DATA 
    Group BY COMMUNITY_AREA_NUMBER
    Order BY count(*) desc
    limit 1
);

 * sqlite:///FinalDB.db
Done.

COMMUNITY_AREA_NAME
Austin


## Author(s)

<h4> Hima Vasudevan </h4>
<h4> Rav Ahuja </h4>
<h4> Ramesh Sannreddy </h4>

## Contribtuor(s)

<h4> Malika Singla </h4>
<h4>Abhishek Gagneja</h4>
<!--
## Change log

| Date       | Version | Changed by        | Change Description                             |
| ---------- | ------- | ----------------- | ---------------------------------------------- |
|2023-10-18  | 2.6     | Abhishek Gagneja  | Modified instruction set |
| 2022-03-04 | 2.5     | Lakshmi Holla     | Changed markdown.                   |
| 2021-05-19 | 2.4     | Lakshmi Holla     | Updated the question                           |
| 2021-04-30 | 2.3     | Malika Singla     | Updated the libraries                          |
| 2021-01-15 | 2.2     | Rav Ahuja         | Removed problem 11 and fixed changelog         |
| 2020-11-25 | 2.1     | Ramesh Sannareddy | Updated the problem statements, and datasets   |
| 2020-09-05 | 2.0     | Malika Singla     | Moved lab to course repo in GitLab             |
| 2018-07-18 | 1.0     | Rav Ahuja         | Several updates including loading instructions |
| 2018-05-04 | 0.1     | Hima Vasudevan    | Created initial version                        |
-->
## <h3 align="center"> © IBM Corporation 2023. All rights reserved. <h3/>
